# Burnback simulation pipeline

Continuation of [Sim](Sim.ipynb)

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from duckdb import sql as sqldf


from Rockets.Simulation.Plots import dots_and_arrows, Interactive_polar, Shape, animate, WebAndPerf
from Rockets.Simulation.Simulation import Lagrangian
from Rockets.Simulation.Pipelines import SimPipeline
from Rockets.Superformula.Formulas import formula1, formula2
from Rockets.Geometry import cart2pol, pol2cart, rad2deg, normals, magn
from Rockets.utils import describe, clip, shape, pick, parse_sf_params

In [2]:
simParams = {
"d": .05,
"steps": 70,
"n": 1000,
"window_size": 70,
"hull_radius": 4,
"interp": 'manydumb'
}


dest = "../../../data/Generated-dbg/"

## Simulation run

[v] Clear Caustics  
[v] Intersection point as replacement for caustic  
[v] Intersection window processes the whole closed circle  
[v] Fill Rarefactions (primitive interpolation)  
[ ] Rarefactions - better interpolation  
[ ] Interpolate multiple point in between rarefactions  
[v] Points stop advancing when front reaches casing  
[ ] Points stop exactly on the casing  
[v] Separation of burning regions  
[ ] Handling cusps - regions that separate entirely from front [?]  
[v] Harvester - gathers data during sim for dbg and visualization  
[v] Interactive plot - animation of burning front  
[ ] Dots and arrows plot - zoom into data of specific steps for debugging  
[ ] Compare results and timings with Kang simulation.  

### Multiple shapes runs 

In [3]:

from pathlib import Path

patt = "*"
path = Path("../../../data/SFs")

paths = list(path.glob(patt +".png"))

tracks = {
    "episodes": {
      "sampling_type": "interval_episodes",
      "sampling_value": 1
    }}


cases = [{"file":
            {"path": p.resolve().as_posix() , "name": p.name, "stem": p.stem}, 
          "SFparams": parse_sf_params(p.stem),
          "simulation": simParams,
          "tracks": tracks,
          "summary": 
            {"case_signature": p.stem}
          } for p in paths ]
   
cases[0]


{'file': {'path': '/home/michael/Studies/DSlab/Capstone/data/SFs/superformula_funcname=sunflower_m=5_a=1_n_1=0.7_n_2=2_s=0_o=1_invert=0_20260512_125111.png',
  'name': 'superformula_funcname=sunflower_m=5_a=1_n_1=0.7_n_2=2_s=0_o=1_invert=0_20260512_125111.png',
  'stem': 'superformula_funcname=sunflower_m=5_a=1_n_1=0.7_n_2=2_s=0_o=1_invert=0_20260512_125111'},
 'SFparams': {'funcname': 'sunflower',
  'm': 5.0,
  'a': 1.0,
  'n_1': 0.7,
  'n_2': 2.0,
  's': 0.0,
  'o': 1.0,
  'invert': 0.0},
 'simulation': {'d': 0.05,
  'steps': 70,
  'n': 1000,
  'window_size': 70,
  'hull_radius': 4,
  'interp': 'manydumb'},
 'tracks': {'episodes': {'sampling_type': 'interval_episodes',
   'sampling_value': 1}},
 'summary': {'case_signature': 'superformula_funcname=sunflower_m=5_a=1_n_1=0.7_n_2=2_s=0_o=1_invert=0_20260512_125111'}}

In [6]:
import warnings

# Ignore all FutureWarnings globally
warnings.filterwarnings("ignore", category=FutureWarning)


from Benchmarking.Benchmarking import Bench
from Benchmarking.utils import isDebugging



bench = Bench(benchmarks_root="/home/michael/Studies/DSlab/Capstone/data/benchmarks",
              output_root="/home/michael/Studies/DSlab/Capstone/data/output",
              folder="20260722/2128",
              onerror = "fail")

simppl = SimPipeline()

# case_template = simppl.case_template()
#case_template["tracks"].pop('profile')

bench.configure(simppl)

bench.set_cases(cases[2:5])
# bench.unfurl_grid(case_template, chosengrid)

bench.run_experiments()


[info] 20260722-214908: Absolute path:  /home/michael/Studies/DSlab/Capstone/data/benchmarks
[info] 20260722-214908: loading 1 files from:/home/michael/Studies/DSlab/Capstone/data/benchmarks/20260722/2128
[info] 20260722-214908: Starting first case in this run:
 experiment
[info] 20260722-214908: 
Out of 3 submitted cases,
  1 cases are already done.
  3 are new and will be run. 
[info] 20260722-214908: Starting first case in this run:
 superformula_funcname=superformula_m=5_a=1_b=1_n_1=0.71_n_2=2.7_n_3=2.7_20260425_205413
[info] 20260722-214908:  initializing .SFparams to {'funcname': 'superformula', 'm': 5, 'a': 1, 'b': 1, 'n_1': 0.71, 'n_2': 2.7, 'n_3': 2.7}
[info] 20260722-214908: Simulation step size(d): 0.05
[info] 20260722-214908: Simulation Steps: 70
[info] 20260722-214908: Curve points (n): 1000
[ping] 20260722-214908: step: 0  | points: (1000,)
[ping] 20260722-214911: step: 39  | points: (770,)                                                                                   

In [ ]:



a = ".track.harvest.curves"

def nunpack(a,n):
    aa = a + [None]*n
    return aa[:n] 

b,c,d,e,f,g = nunpack(a.split("."),6)
b,c,d,e,f,g


('', 'track', 'harvest', None, None, None)

In [ ]:


def simPipeline(Case, dest):

    profile = formula1(**Case["SFparams"])

    simulation = Case["simulation"] 

    SIM = Lagrangian(profile, **simulation)

    # Shape(SIM.R, SIM.T)
    SIM.run(simulation["steps"])
    print("Simulation completed.")

    
    # HSintrsctns = SIM.HSintersections.results()
    # HSsimdata = SIM.HSsim.results()


    WebAndPerf(SIM, dest + Case['stem'] + ".html")

    return SIM
    # dfSim = pd.DataFrame(HSsimdata)
    # dfSim   
    # px.line(dfSim, x = 'SimStep', y = 'C')

In [ ]:
SIM = simPipeline(cases[0], dest)

In [ ]:
animdf = animate(SIM)

In [ ]:
animdf

In [ ]:
animdf.X.notna()

In [ ]:
px.histogram(animdf[animdf.X.notna()].stat)

In [ ]:
WebAndPerf(SIM)

In [ ]:
# from datetime import datetime
# tstp = datetime.now().strftime('%Y-%m-%d %H-%M-%S')
# save_path="dots_arrows_" + tstp + ".html" 

# for Case in cases:
#     SIM = SimPipeline(Case, dest)

In [ ]:


# StepsFilter = (11,12) 
# StepsFilter = (3,4) 
StepsFilter = (4,5,6)
StepsFilter = (31,32,33)

filtr = {"SimStep": StepsFilter}


In [ ]:

dots_and_arrows(SIM, filtr = filtr )

In [ ]:
interp = 'slerp'
SIM2 = SimPipeline(cases[0], dest)

In [ ]:
dots_and_arrows(SIM2, filtr)

In [ ]:
interp = 'manydumb'
d = 0.01
SIM3 = SimPipeline(cases[0], dest)
dots_and_arrows(SIM3, filtr)

In [ ]:
animate(SIM)

In [ ]:
q = """ --
select  *, cast(IsNew as bool)  as isNeww 
from hsdf
where SimStep in (2,3) -- and (I < 10 or I > 700)
-- group by item , tbl
--order by non_null_values desc
"""

wat = sqldf(q).df()

wat